# Configured environment modifiers example

This notebook is a public configured-workflow example for the explicit `temperature_arrhenius_reference` and `ph_gaussian` rate modifiers. It uses artificial framework-benchmark config data created in a temporary notebook output folder and makes no biological claim.

Guardrails: this is not an empirical validation, calibration, literature comparison, or organism-specific physiology example. It adds no fitted response curve, no inferred environment response, and no EnvironmentGrid behavior change. The notebook uses package APIs and configured workflow outputs only; it does not define rate laws, solver logic, or hidden notebook science.


In [ ]:
import csv
import json
import os
import sys
from copy import deepcopy
from pathlib import Path

import yaml

ROOT = Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from fungal_model import run_configured_model

SOURCE_CONFIG = ROOT / "data" / "model_configs" / "toy_homogeneous_ab.yml"
OUTPUT_ROOT = Path(os.environ.get("FUNGMOD_NOTEBOOK_OUTPUT_ROOT", str(ROOT / "notebooks" / "examples" / "Outputs")))
OUTPUT = OUTPUT_ROOT / "17_configured_environment_modifiers_example"
BASE_OUTPUT = OUTPUT / "base"
MODIFIED_OUTPUT = OUTPUT / "modified"
CONFIG = OUTPUT / "configured_inputs" / "toy_homogeneous_environment_modifiers.yml"


## Build an explicit temporary config

The source config is the existing homogeneous software benchmark. The notebook adds explicit artificial parameter records and explicit configured modifier declarations. The package runner still owns model assembly, parameter checking, environment reading, process rates, assumptions, and configured output writing.

The environment comes from the configured environment entity, so temperature and pH are explicit input values. If the pH/temperature values or required parameter records are removed, the configured workflow fails rather than using fallback constants.


In [ ]:
OUTPUT.mkdir(parents=True, exist_ok=True)
CONFIG.parent.mkdir(parents=True, exist_ok=True)

source = "FungMod configured environment-modifier software benchmark."
config = yaml.safe_load(SOURCE_CONFIG.read_text(encoding="utf-8"))
config = deepcopy(config)
config["name"] = "toy homogeneous explicit environment modifiers benchmark"
config["mode"] = "toy"
config["maturity"] = "framework_benchmark"
config["provenance"] = {
    "source": source,
    "measurement_method": "defined software benchmark",
    "confidence_level": "testing",
    "notes": "Artificial configured-workflow demo; not biological environment-response evidence.",
    "validity_range": "framework tests and public examples only",
    "units": "not_applicable",
}

config["parameters"][0]["parameters"].extend(
    [
        {
            "name": "toy Arrhenius activation energy",
            "symbol": "E_a_env",
            "value": 50000.0,
            "units": "joule / mole",
            "uncertainty": 0.0,
            "source": source,
            "confidence_level": "testing",
            "notes": "Artificial framework-benchmark value; not a fitted or biological temperature response.",
            "measurement_method": "defined benchmark value",
            "validity_range": "framework tests only",
        },
        {
            "name": "toy Arrhenius reference temperature",
            "symbol": "T_ref_env",
            "value": 293.15,
            "units": "kelvin",
            "uncertainty": 0.0,
            "source": source,
            "confidence_level": "testing",
            "notes": "Artificial framework-benchmark value; not a fitted or biological temperature response.",
            "measurement_method": "defined benchmark value",
            "validity_range": "framework tests only",
        },
        {
            "name": "toy pH optimum",
            "symbol": "pH_opt_env",
            "value": 6.0,
            "units": "dimensionless",
            "uncertainty": 0.0,
            "source": source,
            "confidence_level": "testing",
            "notes": "Artificial framework-benchmark value; not a fitted or biological pH response.",
            "measurement_method": "defined benchmark value",
            "validity_range": "framework tests only",
        },
        {
            "name": "toy pH width",
            "symbol": "pH_width_env",
            "value": 1.5,
            "units": "dimensionless",
            "uncertainty": 0.0,
            "source": source,
            "confidence_level": "testing",
            "notes": "Artificial framework-benchmark value; not a fitted or biological pH response.",
            "measurement_method": "defined benchmark value",
            "validity_range": "framework tests only",
        },
    ]
)
config["processes"][0]["modifiers"] = [
    {
        "type": "temperature_arrhenius_reference",
        "activation_energy_symbol": "E_a_env",
        "reference_temperature_symbol": "T_ref_env",
        "source": source,
    },
    {
        "type": "ph_gaussian",
        "optimum_symbol": "pH_opt_env",
        "width_symbol": "pH_width_env",
        "source": source,
    },
]

CONFIG.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
{
    "temporary_config": str(CONFIG),
    "modifier_types": [item["type"] for item in config["processes"][0]["modifiers"]],
    "explicit_parameter_symbols": [item["symbol"] for item in config["parameters"][0]["parameters"] if item["symbol"].endswith("_env")],
    "configured_environment_path": config["entities"]["environment"]["path"],
}


## Run the package workflow and inspect emitted outputs

`run_configured_model(...)` validates the explicit symbols, reads the configured environment, builds package process modifiers, and writes the configured output bundle. The comparison to the unmodified benchmark is only a software smoke check that the configured modifiers are active; it is not validation, calibration, empirical comparison, or an inferred pH/temperature response curve.


In [ ]:
base_result = run_configured_model(SOURCE_CONFIG, output_dir=BASE_OUTPUT)
modified_result = run_configured_model(CONFIG, output_dir=MODIFIED_OUTPUT)

metadata = json.loads((MODIFIED_OUTPUT / "configured_metadata.json").read_text(encoding="utf-8"))
input_config = json.loads((MODIFIED_OUTPUT / "input_model_config.json").read_text(encoding="utf-8"))
merged_parameters = json.loads((MODIFIED_OUTPUT / "merged_parameters.json").read_text(encoding="utf-8"))
assumptions = json.loads((MODIFIED_OUTPUT / "assumptions.json").read_text(encoding="utf-8"))
entity_index = json.loads((MODIFIED_OUTPUT / "entity_snapshots" / "index.json").read_text(encoding="utf-8"))
environment_entry = next(entry for entry in entity_index["entities"] if entry["role"] == "environment")
environment = json.loads((MODIFIED_OUTPUT / environment_entry["snapshot_path"]).read_text(encoding="utf-8"))

with (BASE_OUTPUT / "process_rates.csv").open(newline="", encoding="utf-8") as handle:
    base_rates = list(csv.DictReader(handle))
with (MODIFIED_OUTPUT / "process_rates.csv").open(newline="", encoding="utf-8") as handle:
    modified_rates = list(csv.DictReader(handle))

summary = {
    "base_first_rate": float(base_rates[0]["value"]),
    "modified_first_rate": float(modified_rates[0]["value"]),
    "process_rates_changed_by_configured_modifiers": float(modified_rates[0]["value"]) != float(base_rates[0]["value"]),
    "configured_modifier_types": [row["type"] for row in metadata["configured_process_modifiers"]],
    "configured_environment_values": {
        "temperature": environment["temperature"],
        "ph": environment["ph"],
    },
    "explicit_parameter_symbols": [item["symbol"] for item in merged_parameters["parameters"] if item["symbol"].endswith("_env")],
    "assumption_names": [item["name"] for item in assumptions if "scaling" in item["name"] or "pH" in item["name"]],
    "input_modifier_declarations": input_config["processes"][0]["modifiers"],
}

assert modified_result.solver_metadata["success"] is True
assert "a_to_b" in base_result.process_rates
assert "a_to_b" in modified_result.process_rates
assert summary["process_rates_changed_by_configured_modifiers"]
assert {"E_a_env", "T_ref_env", "pH_opt_env", "pH_width_env"}.issubset(summary["explicit_parameter_symbols"])
assert summary["configured_environment_values"]["temperature"]["value"] == 303.15
assert summary["configured_environment_values"]["ph"]["value"] == 7.0
summary


## What this proves and what it does not prove

This example proves that configured generic processes can opt into the package's existing Arrhenius temperature and Gaussian pH modifiers when both the modifier parameter records and the environment temperature/pH values are explicit. It also shows where the configured workflow records those assumptions, limitations, merged parameters, process rates, and modifier metadata.

It does not fit a response curve, calibrate any parameter, compare to empirical data, validate a biological mechanism, infer environment-response biology, alter `EnvironmentGrid` behavior, add oxygen/redox handling, or change solver/model behavior.
